# Tutorial 01: Simplified Active Learning

This tutorial introduces the active learning loop of this code base, on crystalline silicon. A machine-learned
interatomic potential (MLIP) drives a simulation, and the loop repeats four stages until the simulation no longer
meets an atomic environment where the MLIP is uncertain:

- **Stage 1, driver:** run the dynamic driver with the MLIP, and extract the uncertainty of each atom.
- **Stage 2, generate:** generate the structures to label from the uncertain structure of the dynamic driver. This is
  where a diffusion model can be inserted.
- **Stage 3, oracle:** label the generated structures with a reference calculator (the *oracle*), and add them to the
  training data.
- **Stage 4, train:** retrain the MLIP on the new training data.

A calculation can easily be resumed from any stage: by default, it resumes just after the last stage that completed
successfully.

This tutorial follows `examples/01_activelearning_mtp_md_sw_noop.py` and `examples/02_activelearning_mtp_artn_sw_noop.py`.
It is *simplified* because it uses a cheap oracle, Stillinger-Weber instead of DFT, and no diffusion model: the
uncertain structure is labelled as is, without being excised and repainted (see tutorial 03). The components are:

- **MLIP:** a Moment Tensor Potential (MTP), fitted by MLIP-3, whose extrapolation grade gives the uncertainty of each atom.
- **Dynamic driver:** molecular dynamics (example 01), or ARTn saddle point searches (example 02), run by LAMMPS.
- **Oracle:** Stillinger-Weber.
- **Sample maker:** no-op, which returns the uncertain structure itself.

> **Note**
>
> This tutorial calls three external programs: a LAMMPS with the `mtp/extrapolation` pair style and the PLUGIN
> package ([lammps-mtp-kokkos](https://github.com/RichardZJM/lammps-mtp-kokkos)), the `mlp` executable of
> [MLIP-3](https://gitlab.com/ashapeev/mlip-3), and the [ARTn plugin](https://gitlab.com/mammasmias/artn-plugin)
> (only for the ARTn dynamic driver).
>
> Before launching Jupyter, define the following environment variables in your terminal (or add them to your
> `~/.bashrc` or `~/.zshrc`):
>
> ```bash
> export DFMSMD_LAMMPS_PATH=Replace_with_absolute_path_to_lmp                          # Change this line
> export DFMSMD_MLP_PATH=Replace_with_absolute_path_to_mlp                             # Change this line
> export DFMSMD_ARTN_LIBRARY_PLUGIN_PATH=Replace_with_absolute_path_to_libartn-lmp.so  # Change this line
> ```
>
> Then launch Jupyter from that same terminal: a notebook only sees the environment variables of the shell that
> started Jupyter.

In [ ]:
import warnings

# To suppress warnings for clearer output
warnings.simplefilter("ignore")

In [ ]:
# Define the directory where artifacts will be written. Delete it if it exists to start clean.
import shutil
from diffusion_for_multi_scale_molecular_dynamics import TOP_DIR

output_path = TOP_DIR / "tutorials" / "output" / "01-simplified_active_learning"

# Restart from an empty directory
if output_path.is_dir():
    shutil.rmtree(output_path)

output_path.mkdir(parents=True, exist_ok=True)

# Stage 1, driver: Molecular Dynamics

The `DynamicDriver` leads the Driver stage, or one of its child classes (`MdDriver`, `ArtnDriver`): it explores new
configurations with the current MLIP. Here, the `MdDriver` runs with LAMMPS a molecular dynamics of a 512-atom diamond
silicon supercell (NVT at 300 K, 1 fs time step, 1000 steps). At every step, the MTP computes the uncertainty of each
atom; as soon as one is above the uncertainty threshold of the campaign (see Active Learning below), the simulation
stops and the structure at that step becomes the uncertain structure of the round. When the 1000 steps complete
without stopping, the campaign is complete.

An active learning campaign produces a lot of files. We point to the main ones stage by stage; they are all written in
the campaign directory (`output_path / "md"`, see Active Learning below):

- `provided_configurations.traj`: the starting training data, written once at the start of the campaign (see Stage 3).
- `epoch_N/dynamic.traj`: the uncertain structure of round N, where the dynamic driver stopped.
- `epoch_N/dynamic/`: the working directory of LAMMPS for round N (input, log and trajectory dumps).

The LAMMPS runner defined here is shared by every component that calls LAMMPS: the dynamic driver, the oracle and the
MLIP.

In [ ]:
import os
from pathlib import Path

from ase.build import bulk

from diffusion_for_multi_scale_molecular_dynamics.dynamic_driver.md_driver.md_driver import MdDriver
from diffusion_for_multi_scale_molecular_dynamics.oracle.lammps_runner import SubprocessLammpsRunner

lammps_runner = SubprocessLammpsRunner(lammps_executable_path=Path(os.environ["DFMSMD_LAMMPS_PATH"]),
                                       mpi_processors=1,
                                       openmp_threads=1,
                                       mpi_executable="mpirun")

# The (orthogonal) starting configuration of the dynamic driver: 512 atoms of diamond silicon.
initial_configuration = bulk("Si", "diamond", a=5.43, cubic=True).repeat((4, 4, 4))

dynamic_driver = MdDriver(lammps_runner=lammps_runner,
                          initial_configuration=initial_configuration,
                          temperature=300.0,
                          timestep=0.001,
                          number_of_steps=1000)

# Stage 2, generate: Sample Maker

The `BaseSampleMaker` leads the Generate stage, or one of its child classes (`NoOpSampleMaker`,
`ExciseAndNoOpSampleMaker`, `ExciseAndRandomSampleMaker`, `ExciseAndRepaintSampleMaker`). This stage turns the
uncertain structure from the dynamic driver into structures ready to be labelled by the oracle. First, the atom
selector picks the uncertain atoms, those whose extrapolation grade is above the threshold. A sample maker could then
excise the environment of each selected atom and repaint the rest of a new cell with a diffusion model (see tutorial
03). Here, the `NoOpSampleMaker` returns the uncertain structure itself, so the oracle labels the full 512-atom
structure.

- `epoch_N/generate.traj`: the structures generated in round N, not labelled yet.

In [ ]:
from diffusion_for_multi_scale_molecular_dynamics.sample_maker.atom_selector.threshold_atom_selector import (
    ThresholdAtomSelector, ThresholdAtomSelectorParameters)
from diffusion_for_multi_scale_molecular_dynamics.sample_maker.no_op_sample_maker import (
    NoOpSampleMaker, NoOpSampleMakerArguments)

# Atoms with an MTP extrapolation grade (gamma) above 2 are uncertain.
atom_selector = ThresholdAtomSelector(ThresholdAtomSelectorParameters(uncertainty_threshold=2.0))
sample_maker = NoOpSampleMaker(sample_maker_arguments=NoOpSampleMakerArguments(element_list=["Si"]),
                               atom_selector=atom_selector)

# Stage 3, oracle: Stillinger-Weber

The `BaseSinglePointCalculator` leads the Oracle stage, or one of its child classes (`LammpsSinglePointCalculator`,
`AbinitSinglePointCalculator`, `FlareSinglePointCalculator`): it labels structures with their reference energy and
forces. Here, the `LammpsSinglePointCalculator` evaluates with LAMMPS the Stillinger-Weber potential for crystalline
silicon (shipped with the package).

Before the campaign, the oracle also labels the starting training data of the MTP: a 16-atom cell of diamond
silicon. At equilibrium, its forces vanish.

- `epoch_N/oracle.traj`: the structures of round N with their energy and forces, added to the training data.
- `epoch_N/oracle/`: the working directory of the oracle for round N.

In [ ]:
from diffusion_for_multi_scale_molecular_dynamics.io.lammps.potential.stillinger_weber import \
    StillingerWeberPotential
from diffusion_for_multi_scale_molecular_dynamics.oracle import SW_COEFFICIENTS_DIR
from diffusion_for_multi_scale_molecular_dynamics.oracle.lammps_single_point_calculator import \
    LammpsSinglePointCalculator
from diffusion_for_multi_scale_molecular_dynamics.utils.structure_utils import label_configurations

potential = StillingerWeberPotential(sw_coefficients_file_path=SW_COEFFICIENTS_DIR / "Si.sw")
oracle = LammpsSinglePointCalculator(lammps_potential=potential, lammps_runner=lammps_runner)

# The starting training data of the MTP: a 16-atom cell of diamond silicon, labelled by the oracle.
provided_configurations = label_configurations([bulk("Si", "diamond", a=5.43).repeat((2, 2, 2))], oracle)

print(f"Energy: {provided_configurations[0].get_potential_energy():.4f} eV")
print(f"Largest force component: {abs(provided_configurations[0].get_forces()).max():.2e} eV/Å")

# Stage 4, train: Moment Tensor Potential

The `BaseMLIP` leads the Train stage, or one of its child classes (`MtpMlip`, `FlareMLIP`, `GraceMlip`): it fits the
MLIP on the training data. Here, the `MtpMlip` is a Moment Tensor Potential (MTP) of level 6 with a 5 Å cutoff, fitted
by the `mlp` executable of MLIP-3 on the energies and forces of every labelled structure (no stress). After each
round, it is fitted again on the training data, which now includes the structures labelled by the oracle.

The Train stage writes three files:

- `precomputation/`: the initial MTP (round 0), fitted on `first_training.traj`, built from the provided
  configurations.
- `epoch_N/model/`: the MTP fitted in round N.
- `latest_mlip`: a link to the most recent MTP.

MTP and GRACE measure their uncertainty with the D-optimality criterion. Among the atomic environments of the training
data, D-optimality selects those whose descriptors span the largest volume, i.e. maximize the determinant of the
matrix they form: the most diverse environments, which best constrain the parameters of the model. The extrapolation
grade γ of an atom expresses its environment in terms of these selected environments: below γ = 1 the model
interpolates, above it extrapolates. From the training data, they construct this active learning set, which must
contain at least as many atomic environments as the model has parameters, so that the matrix is square. If the
provided configurations do not hold enough atomic environments, perturbed copies are automatically generated and
labelled by the oracle, and added to `first_training.traj`.

In [ ]:
from diffusion_for_multi_scale_molecular_dynamics.mlip.mtp.mtp_configuration import MtpConfiguration
from diffusion_for_multi_scale_molecular_dynamics.mlip.mtp.mtp_mlip import MtpMlip
from diffusion_for_multi_scale_molecular_dynamics.mlip.mtp.mtp_trainer import MtpTrainer

mlp_executable_path = Path(os.environ["DFMSMD_MLP_PATH"])

mtp_configuration = MtpConfiguration(elements=["Si"], level=6, max_dist=5.0,
                                     energy_weight=1.0, force_weight=0.01, stress_weight=0.0, site_en_weight=0.0)
mtp_trainer = MtpTrainer(mtp_configuration=mtp_configuration, mlp_executable_path=mlp_executable_path)
mlip = MtpMlip(mtp_trainer=mtp_trainer, lammps_runner=lammps_runner)

# Active Learning

`ActiveLearning` puts the four components together, and `run_campaign` runs the loop. Before the first round, it
fits an initial MTP on the provided configurations (round 0). Then each round goes through the four stages, until the
molecular dynamics completes without meeting an uncertain atom, or after `maximum_number_of_rounds` rounds.

With `restart_from_stage="auto"`, running this cell again on an existing directory resumes just after the last stage
that completed; a stage name forces the resume stage of the latest round instead.

The campaign should last about 10-15 epochs and take about 2 to 5 minutes to run.

In [ ]:
from diffusion_for_multi_scale_molecular_dynamics.active_learning_loop.active_learning import ActiveLearning

active_learning = ActiveLearning(oracle_single_point_calculator=oracle,
                                 sample_maker=sample_maker,
                                 dynamic_driver=dynamic_driver,
                                 mlip=mlip)

active_learning.run_campaign(uncertainty_threshold=2.0,  # MTP extrapolation grade (gamma) that stops the dynamic driver
                             working_directory=output_path / "md",
                             provided_configurations=provided_configurations,
                             maximum_number_of_rounds=100,
                             restart_from_stage="auto")  # either {"auto", "driver", "generate", "oracle", "train"}

Besides the files of each stage, the campaign writes two files in its directory:

- `active_learning.log`: the log of every stage of every round, each line prefixed with its epoch and stage (for
  example `Epoch 3 Driver`). It is the first place to look to follow a campaign, or to find why it stopped.
- `campaign_details.yaml`: written at the end of the campaign, a summary of the final MTP: the uncertainty threshold,
  the last epoch, the number of training configurations and atomic environments, and the training RMSE of the
  energies and forces.

# Analysis

Once the campaign is complete, we can analyze the run.

**Molecular dynamics steps per epoch.** The number of steps the molecular dynamics ran at each epoch before an
uncertain atom stopped it. As the MTP learns, the dynamics should run longer, until it completes the 1000 steps
without interruption. It is an orange flag if the simulation does few steps every epoch, and then completes the
simulation in one long epoch without interruption.

In [ ]:
import re

from matplotlib import pyplot as plt

campaign_directory = output_path / "md"
epoch_directories = sorted(campaign_directory.glob("epoch_*"), key=lambda path: int(path.name.removeprefix("epoch_")))
epochs = [int(path.name.removeprefix("epoch_")) for path in epoch_directories]

# LAMMPS ends log.lammps with the number of steps it ran: "Loop time of ... for 637 steps with 512 atoms".
number_of_steps = []
for epoch_directory in epoch_directories:
    lammps_log = (epoch_directory / "dynamic" / "log.lammps").read_text()
    number_of_steps.append(int(re.search(r"Loop time of .* for (\d+) steps", lammps_log).group(1)))

plt.plot(epochs, number_of_steps, "o-")
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Molecular dynamics steps")
plt.show()

**Energy of the labelled structure per epoch.** The energy per atom computed by the oracle for the structure labelled
at each epoch (with the no-op sample maker, the uncertain structure itself), compared with the energy of the crystal
at equilibrium. For molecular dynamics at 300 K, a harmonic crystal holds on average 3/2 k<sub>B</sub>T ≈ 39 meV/atom
of potential energy, so the labelled structures should lie between the crystal and a few tens of meV/atom above it.
It is a red flag if the energy is far above (hundreds of meV/atom) or keeps increasing from one epoch to the next: the
MTP drives the dynamics into unphysical configurations, such as atoms too close to each other, and the training data
fills with structures the dynamics should never visit. This rule only holds for molecular dynamics: ARTn climbs on
purpose to saddle points, above the thermal energy, so its labelled structures do not follow it.

In [ ]:
import ase.io

# The labelled structures of each epoch, with their oracle energy, are in epoch_N/oracle.traj. The last epoch
# completes its dynamics: it has no structure to label, hence no oracle.traj.
labelled_epochs, energies_per_atom = [], []
for epoch, epoch_directory in zip(epochs, epoch_directories):
    if (epoch_directory / "oracle.traj").is_file():
        for atoms in ase.io.read(epoch_directory / "oracle.traj", index=":"):
            labelled_epochs.append(epoch)
            energies_per_atom.append(atoms.get_potential_energy() / len(atoms))

crystal = provided_configurations[0]
plt.plot(labelled_epochs, energies_per_atom, "o-", label="Labelled structure")
plt.axhline(crystal.get_potential_energy() / len(crystal), color="gray", linestyle="--", label="Crystal at equilibrium")
plt.xlabel("Epoch")
plt.ylabel("Oracle energy (eV/atom)")
plt.legend()
plt.show()

**Temperature during the last dynamics.** The temperature at every step of the molecular dynamics of the last epoch,
the one that completes its 1000 steps. The Nosé-Hoover thermostat keeps it around 300 K. It is a red flag if the
temperature climbs well above 300 K: the MTP forces then create energy instead of conserving it, a sign that the
dynamics runs where the MTP is wrong.

The temperature also oscillates strongly. The velocities start at 300 K, with every atom exactly on its crystal site,
at the bottom of its potential well. About half of this kinetic energy then flows into potential energy, so the
temperature first drops before the thermostat brings it back. Since all the atoms start at the bottom of their well
at the same time, all the vibration modes oscillate in phase: kinetic and potential energy are exchanged together, and
the temperature oscillates with a period of about 30 fs (30 steps), half the period of the optical phonon of silicon
(about 15.5 THz). The crystal at 300 K is nearly harmonic, so these modes dephase slowly, and the oscillation is still
about ±50 K after 1000 steps, far above the ±10 K expected at equilibrium for 512 atoms. A thermalization before the
campaign (for example, a short molecular dynamics, or random displacements of the atoms, to start from a structure
already at 300 K), or a longer molecular dynamics, would have helped here.

In [ ]:
from itertools import takewhile

import numpy as np

# The thermodynamic output of LAMMPS is a table between its header line ("Step PotEng Temp v_max_unc") and the
# "Loop time" line.
lammps_log_lines = (epoch_directories[-1] / "dynamic" / "log.lammps").read_text().splitlines()
header_index = next(index for index, line in enumerate(lammps_log_lines) if line.split()[:1] == ["Step"])
columns = lammps_log_lines[header_index].split()
table_lines = takewhile(lambda line: not line.startswith("Loop time"), lammps_log_lines[header_index + 1:])
thermo = np.array([line.split() for line in table_lines if line.strip() and line.split()[0].isdigit()], dtype=float)

plt.plot(thermo[:, columns.index("Step")], thermo[:, columns.index("Temp")])
plt.axhline(300.0, color="gray", linestyle="--", label="Thermostat temperature")
plt.xlabel("Molecular dynamics step")
plt.ylabel("Temperature (K)")
plt.legend()
plt.show()

# Dynamic Driver: ARTn

Molecular dynamics at 300 K mostly visits the neighborhood of the crystal. The activation-relaxation technique
nouveau (ARTn) explores further: it pushes atoms out of a local minimum, climbs along the lowest curvature direction
up to a saddle point, then relaxes into the neighboring minimum. These transition states are rarely reached by
molecular dynamics, but they control the diffusion of defects.

Here, we do the same campaign but with a different dynamic driver: the oracle, the sample maker and the MTP configuration are the same.
This campaign starts from a fresh MTP, in its own directory. An uncertain atom interrupts the search as in molecular
dynamics, and the saddle points found are counted over all rounds: the campaign is complete once ARTn has found 50 of
them.

The campaign should last about 30 epochs and take about 30 minutes to run.

In [ ]:
from diffusion_for_multi_scale_molecular_dynamics.dynamic_driver.artn_driver.artn_driver import ArtnDriver

# artn.in uses the default parameters of ArtnInputConfiguration; see the ARTn documentation for each variable.
artn_driver = ArtnDriver(lammps_runner=lammps_runner,
                         initial_configuration=initial_configuration,
                         artn_library_plugin_path=Path(os.environ["DFMSMD_ARTN_LIBRARY_PLUGIN_PATH"]),
                         number_of_requested_saddles=50)

# A fresh MTP, so that this campaign starts from the same point as the molecular dynamics one.
artn_mlip = MtpMlip(mtp_trainer=MtpTrainer(mtp_configuration=mtp_configuration,
                                           mlp_executable_path=mlp_executable_path),
                    lammps_runner=lammps_runner)

artn_active_learning = ActiveLearning(oracle_single_point_calculator=oracle,
                                      sample_maker=sample_maker,
                                      dynamic_driver=artn_driver,
                                      mlip=artn_mlip)

artn_active_learning.run_campaign(uncertainty_threshold=2.0,
                                  working_directory=output_path / "artn",
                                  provided_configurations=provided_configurations,
                                  maximum_number_of_rounds=100,
                                  restart_from_stage="auto")